# Official SuperNEMO Transformer Training

Runs the frozen three-tokenization × two-position-encoding matrix for `2nu=1` versus `Bi214=0`. The notebook trains and evaluates directly in the selected kernel. The shared event reconstruction, Wing split manifest, and validation checkpoint-selection protocol remain authoritative; evaluation uses the repository's fingerprinted EnergyBench snapshot.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys

import pandas as pd
import torch

configured_root = os.environ.get('SUPERNEMO_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([Path.cwd(), *Path.cwd().parents])
REPOSITORY_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'supernemo_detector' / 'supernemobench').is_dir()
     and (candidate / 'supernemo_detector' / 'supernemo_transformer').is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError(
        'Could not locate the repository; set SUPERNEMO_TRANSFORMER_PROJECT_ROOT.'
    )
DETECTOR_ROOT = REPOSITORY_ROOT / 'supernemo_detector'
if str(DETECTOR_ROOT) not in sys.path:
    sys.path.insert(0, str(DETECTOR_ROOT))

print('Repository:', REPOSITORY_ROOT)
print('Python:', Path(sys.executable).resolve())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Repository: /home/klz/Data/zeronu_benchmark/Transformer_Approach
Python: /home/liuser/miniconda3/envs/zeronu-next/bin/python3.11
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


In [2]:
DATA_ROOT = Path(os.environ.get(
    'SUPERNEMO_DATA_ROOT',
    '/home/klz/Data/zeronu_benchmark/SuperNEMO',
)).expanduser()
SPLIT_MANIFEST = Path(os.environ.get(
    'SUPERNEMO_SPLIT_MANIFEST',
    '/home/wenyu/SuperNEMO/data/manifests/split_manifest.json',
)).expanduser()
EVALUATION_MANIFEST = Path(os.environ.get(
    'SUPERNEMO_EVALUATION_MANIFEST',
    '/home/wenyu/SuperNEMO/evaluation/supernemo_2nu_vs_bi214.json',
)).expanduser()
ENERGYBENCH_PACKAGE = Path(os.environ.get(
    'SUPERNEMO_ENERGYBENCH_PACKAGE',
    str(REPOSITORY_ROOT / 'exo200_detector' / 'frozen_energybench' / 'energybench'),
)).expanduser()
os.environ['SUPERNEMO_SPLIT_MANIFEST'] = str(SPLIT_MANIFEST)
os.environ['SUPERNEMO_EVALUATION_MANIFEST'] = str(EVALUATION_MANIFEST)
os.environ['SUPERNEMO_ENERGYBENCH_PACKAGE'] = str(ENERGYBENCH_PACKAGE)

for name, required in {
    'data root': DATA_ROOT,
    'split manifest': SPLIT_MANIFEST,
    'evaluation manifest': EVALUATION_MANIFEST,
    'frozen EnergyBench package': ENERGYBENCH_PACKAGE,
}.items():
    if not required.exists():
        raise FileNotFoundError(f'Missing {name}: {required}')

from supernemobench.config import (
    ARCHITECTURES,
    DEFAULT_OUTPUT_ROOT,
    TRANSFORMER_ARCHITECTURE_IDS,
)
from supernemobench.workflow import main as run_workflow

OUTPUT_ROOT = DEFAULT_OUTPUT_ROOT / 'classification'
DEVICE = os.environ.get('SUPERNEMO_DEVICE', '').strip()
requested = os.environ.get('SUPERNEMO_RUN_IDS', '').strip()
SELECTED_RUN_IDS = (
    tuple(value.strip() for value in requested.split(',') if value.strip())
    if requested else TRANSFORMER_ARCHITECTURE_IDS
)
unknown = set(SELECTED_RUN_IDS) - set(TRANSFORMER_ARCHITECTURE_IDS)
if unknown:
    raise ValueError(f'Unknown SUPERNEMO_RUN_IDS: {sorted(unknown)}')

print('Dataset:', DATA_ROOT)
print('Split manifest:', SPLIT_MANIFEST)
print('Evaluation manifest:', EVALUATION_MANIFEST)
print('Frozen EnergyBench:', ENERGYBENCH_PACKAGE)
print('Outputs:', OUTPUT_ROOT)
print('Device override:', DEVICE or 'workflow default')
print('Selected runs:', list(SELECTED_RUN_IDS))

Dataset: /home/klz/Data/zeronu_benchmark/SuperNEMO
Split manifest: /home/wenyu/SuperNEMO/data/manifests/split_manifest.json
Evaluation manifest: /home/wenyu/SuperNEMO/evaluation/supernemo_2nu_vs_bi214.json
Frozen EnergyBench: /home/klz/Data/zeronu_benchmark/Transformer_Approach/exo200_detector/frozen_energybench/energybench
Outputs: /home/klz/Data/zeronu_benchmark/Transformer_Approach/supernemo_detector/outputs/classification
Device override: workflow default
Selected runs: ['transformer_001_entity_coordinate_mlp', 'transformer_002_patch_88mm_coordinate_mlp', 'transformer_003_patch_88mm_fourier_xyz', 'transformer_004_entity_fourier_xyz', 'transformer_005_summary_16_coordinate_mlp', 'transformer_006_summary_16_fourier_xyz']


In [3]:
experiment_table = pd.DataFrame([
    {
        'run_id': model_id,
        'tokenization': ARCHITECTURES[model_id].tokenization.tokenization,
        'position_encoding': ARCHITECTURES[model_id].model['position_encoding'],
        'feature_dim': ARCHITECTURES[model_id].model['feature_dim'],
        'batch_size': ARCHITECTURES[model_id].training.batch_size,
        'learning_rate': ARCHITECTURES[model_id].training.learning_rate,
        'workers': ARCHITECTURES[model_id].training.num_workers,
        'maximum_epochs': ARCHITECTURES[model_id].training.epochs,
        'early_stopping_patience': ARCHITECTURES[model_id].training.early_stopping_patience,
        'checkpoint_selection': 'validation energy-matched AUC',
    }
    for model_id in SELECTED_RUN_IDS
])
display(experiment_table)

,run_id,tokenization,position_encoding,feature_dim,batch_size,learning_rate,workers,maximum_epochs,early_stopping_patience,checkpoint_selection
0,transformer_001_entity_coordinate_mlp,entity,coordinate_mlp,4,64,0.0005,8,50,5,validation energy-matched AUC
1,transformer_002_patch_88mm_coordinate_mlp,patch,coordinate_mlp,8,64,0.0005,8,50,5,validation energy-matched AUC
2,transformer_003_patch_88mm_fourier_xyz,patch,fourier_xyz,8,64,0.0005,8,50,5,validation energy-matched AUC
3,transformer_004_entity_fourier_xyz,entity,fourier_xyz,4,64,0.0005,8,50,5,validation energy-matched AUC
4,transformer_005_summary_16_coordinate_mlp,summary,coordinate_mlp,8,64,0.0005,8,50,5,validation energy-matched AUC
5,transformer_006_summary_16_fourier_xyz,summary,fourier_xyz,8,64,0.0005,8,50,5,validation energy-matched AUC


In [4]:
def workflow_arguments(model_id, mode):
    arguments = [
        '--task', 'classification',
        '--model', model_id,
        '--mode', mode,
        '--data-root', str(DATA_ROOT),
        '--manifest-path', str(SPLIT_MANIFEST),
    ]
    if DEVICE:
        arguments.extend(['--device', DEVICE])
    return arguments

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def training_is_complete(run_dir):
    marker_path = run_dir / 'training_complete.json'
    last_path = run_dir / 'last.pt'
    best_path = run_dir / 'best.pt'
    if not all(path.is_file() for path in (marker_path, last_path, best_path)):
        return False
    marker = json.loads(marker_path.read_text(encoding='utf-8'))
    return (
        marker.get('architecture_id') == run_dir.name
        and marker.get('last_checkpoint_sha256') == file_sha256(last_path)
        and marker.get('best_checkpoint_sha256') == file_sha256(best_path)
        and marker.get('stop_reason') in {'epoch_limit', 'early_stopping'}
    )

def testing_is_complete(run_dir):
    if not training_is_complete(run_dir):
        return False
    metrics_path = run_dir / 'test_evaluation' / 'test_metrics.json'
    if not metrics_path.is_file():
        return False
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    expected_model_id = f"{run_dir.name}@{file_sha256(run_dir / 'best.pt')[:12]}"
    return metrics.get('energybench_model_id') == expected_model_id

for model_id in SELECTED_RUN_IDS:
    run_dir = OUTPUT_ROOT / model_id
    print('\n' + '=' * 96)
    print(model_id)
    print('=' * 96)

    if training_is_complete(run_dir):
        print('Skipping completed training run.')
    else:
        if run_dir.is_dir() and any(run_dir.iterdir()):
            raise RuntimeError(
                f'Incomplete run artifacts in {run_dir}; archive the directory before a clean restart.'
            )
        print('Starting a clean training run in the notebook kernel...', flush=True)
        run_workflow(workflow_arguments(model_id, 'train'))

    if testing_is_complete(run_dir):
        print('Skipping completed held-out evaluation.')
    else:
        print('Running held-out EnergyBench evaluation...', flush=True)
        run_workflow(workflow_arguments(model_id, 'test'))


transformer_001_entity_coordinate_mlp
Starting a clean training run in the notebook kernel...


KeyboardInterrupt: 

In [ ]:
rows = []
for model_id in SELECTED_RUN_IDS:
    run_dir = OUTPUT_ROOT / model_id
    metrics_path = run_dir / 'test_evaluation' / 'test_metrics.json'
    if not metrics_path.is_file():
        continue
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    checkpoint = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
    rows.append({
        'run_id': model_id,
        'tokenization': ARCHITECTURES[model_id].tokenization.tokenization,
        'position_encoding': ARCHITECTURES[model_id].model['position_encoding'],
        'best_epoch': checkpoint['epoch'],
        'best_validation_energy_matched_auc': checkpoint['score'],
        'test_energy_matched_auc': metrics.get('energy_matched_auc'),
        'test_auc': metrics.get('auc'),
        'test_accuracy': metrics.get('accuracy'),
        'events': metrics.get('events'),
    })
results = pd.DataFrame(rows)
print('Completed official runs:', len(results))
display(results)